In [1]:
import cv2
import numpy as np
import pandas as pd
import time
import os
from google.colab import files

In [2]:
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Uploaded video:", video_path)

Saving excessive_movement.mp4 to excessive_movement.mp4
Uploaded video: excessive_movement.mp4


In [5]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Video could not be opened")

fps_input = cap.get(cv2.CAP_PROP_FPS)
if fps_input == 0:
    fps_input = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

base_name = os.path.splitext(video_path)[0]

output_path = base_name + "_member4_output.mp4"
log_path = base_name + "_member4_log.csv"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps_input, (width, height))

background_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

previous_gray = None
frame_number = 0
violation_logs = []

excessive_start = None
camera_blocked_start = None
violation_count = 0

start_time_total = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_start_time = time.time()
    frame_number += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (21, 21), 0)

    fg_mask = background_subtractor.apply(blur)

    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5, 5), np.uint8)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_DILATE, kernel)

    frame_diff_score = 0

    if previous_gray is not None:
        diff = cv2.absdiff(previous_gray, blur)
        _, diff_thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
        frame_diff_score = np.sum(diff_thresh > 0) / (width * height)

    previous_gray = blur.copy()

    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    total_motion_area = 0
    largest_box = None

    for contour in contours:
        area = cv2.contourArea(contour)

        if area > 800:
            x, y, w, h = cv2.boundingRect(contour)
            total_motion_area += area

            if largest_box is None or area > largest_box[4]:
                largest_box = (x, y, w, h, area)

    motion_ratio = total_motion_area / (width * height)
    brightness = np.mean(gray)

    violation = "Normal"
    risk_score = 0
    risk_level = "Low"

    current_time = frame_number / fps_input

    # -----------------------------
    # Excessive Movement Detection
    # -----------------------------
    if motion_ratio > 0.08 or frame_diff_score > 0.07:
        if excessive_start is None:
            excessive_start = current_time

        if current_time - excessive_start >= 0.5:
            violation = "Excessive Movement Detected"

            if motion_ratio > 0.20:
                risk_score = 40
                risk_level = "High"
            elif motion_ratio > 0.12:
                risk_score = 25
                risk_level = "Medium"
            else:
                risk_score = 15
                risk_level = "Low"
    else:
        excessive_start = None

    # -----------------------------
    # Camera Blocked Detection
    # -----------------------------
    if brightness < 35:
        if camera_blocked_start is None:
            camera_blocked_start = current_time

        if current_time - camera_blocked_start >= 1.0:
            violation = "Camera Blocked"
            risk_score = 50
            risk_level = "High"
    else:
        camera_blocked_start = None

    # -----------------------------
    # Dummy placeholders for team integration
    # Later these will come from Members 1, 2, and 3
    # -----------------------------
    face_alert = False
    phone_alert = False
    gaze_alert = False

    final_score = risk_score

    if face_alert:
        final_score += 30

    if phone_alert:
        final_score += 50

    if gaze_alert:
        final_score += 20

    # -----------------------------
    # Fixed Overall Status Logic
    # -----------------------------
    if final_score >= 50:
        overall_status = "Violation"
    elif final_score >= 20 or violation != "Normal":
        overall_status = "Warning"
    else:
        overall_status = "Normal"

    if violation != "Normal":
        violation_count += 1

    # Draw bounding box around largest moving region
    if largest_box is not None:
        x, y, w, h, area = largest_box
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    latency_ms = (time.time() - frame_start_time) * 1000

    # -----------------------------
    # Text overlays on output video
    # -----------------------------
    cv2.putText(frame, f"Status: {violation}", (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.putText(frame, f"Overall: {overall_status}", (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    cv2.putText(frame, f"Risk Score: {final_score} ({risk_level})", (30, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Motion Ratio: {motion_ratio:.3f}", (30, 160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Latency: {latency_ms:.2f} ms", (30, 200),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Violation Count: {violation_count}", (30, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Save log only when a violation happens
    if violation != "Normal":
        violation_logs.append({
            "timestamp_seconds": round(current_time, 2),
            "frame_number": frame_number,
            "violation_type": violation,
            "overall_status": overall_status,
            "risk_score": final_score,
            "risk_level": risk_level,
            "motion_ratio": round(motion_ratio, 4),
            "frame_difference_score": round(frame_diff_score, 4),
            "brightness": round(brightness, 2),
            "latency_ms": round(latency_ms, 2)
        })

    out.write(frame)

cap.release()
out.release()

total_time = time.time() - start_time_total
processed_fps = frame_number / total_time

df = pd.DataFrame(violation_logs)
df.to_csv(log_path, index=False)

print("Processing complete")
print("Input video:", video_path)
print("Total frames processed:", frame_number)
print("Average FPS:", round(processed_fps, 2))
print("Total violations:", violation_count)
print("Violation log saved as:", log_path)
print("Output video saved as:", output_path)

Processing complete
Input video: excessive_movement.mp4
Total frames processed: 469
Average FPS: 17.23
Total violations: 41
Violation log saved as: excessive_movement_member4_log.csv
Output video saved as: excessive_movement_member4_output.mp4


In [6]:
df = pd.read_csv(log_path)
df.head(), len(df)

(   timestamp_seconds  frame_number               violation_type  \
 0               3.20            96  Excessive Movement Detected   
 1               3.23            97  Excessive Movement Detected   
 2               3.27            98  Excessive Movement Detected   
 3               3.30            99  Excessive Movement Detected   
 4               3.33           100  Excessive Movement Detected   
 
   overall_status  risk_score risk_level  motion_ratio  frame_difference_score  \
 0        Warning          15        Low        0.1019                  0.0000   
 1        Warning          25     Medium        0.1251                  0.1705   
 2        Warning          25     Medium        0.1227                  0.0740   
 3        Warning          15        Low        0.1133                  0.1107   
 4        Warning          15        Low        0.1146                  0.0531   
 
    brightness  latency_ms  
 0      112.47       43.66  
 1      112.03       46.05  
 2      1

In [7]:
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Uploaded video:", video_path)

Saving normal_sitting.mp4 to normal_sitting.mp4
Uploaded video: normal_sitting.mp4


In [8]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Video could not be opened")

fps_input = cap.get(cv2.CAP_PROP_FPS)
if fps_input == 0:
    fps_input = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

base_name = os.path.splitext(video_path)[0]

output_path = base_name + "_member4_output.mp4"
log_path = base_name + "_member4_log.csv"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps_input, (width, height))

background_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

previous_gray = None
frame_number = 0
violation_logs = []

excessive_start = None
camera_blocked_start = None
violation_count = 0

start_time_total = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_start_time = time.time()
    frame_number += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (21, 21), 0)

    fg_mask = background_subtractor.apply(blur)

    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5, 5), np.uint8)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_DILATE, kernel)

    frame_diff_score = 0

    if previous_gray is not None:
        diff = cv2.absdiff(previous_gray, blur)
        _, diff_thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
        frame_diff_score = np.sum(diff_thresh > 0) / (width * height)

    previous_gray = blur.copy()

    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    total_motion_area = 0
    largest_box = None

    for contour in contours:
        area = cv2.contourArea(contour)

        if area > 800:
            x, y, w, h = cv2.boundingRect(contour)
            total_motion_area += area

            if largest_box is None or area > largest_box[4]:
                largest_box = (x, y, w, h, area)

    motion_ratio = total_motion_area / (width * height)
    brightness = np.mean(gray)

    violation = "Normal"
    risk_score = 0
    risk_level = "Low"

    current_time = frame_number / fps_input

    # -----------------------------
    # Excessive Movement Detection
    # -----------------------------
    if motion_ratio > 0.08 or frame_diff_score > 0.07:
        if excessive_start is None:
            excessive_start = current_time

        if current_time - excessive_start >= 0.5:
            violation = "Excessive Movement Detected"

            if motion_ratio > 0.20:
                risk_score = 40
                risk_level = "High"
            elif motion_ratio > 0.12:
                risk_score = 25
                risk_level = "Medium"
            else:
                risk_score = 15
                risk_level = "Low"
    else:
        excessive_start = None

    # -----------------------------
    # Camera Blocked Detection
    # -----------------------------
    if brightness < 35:
        if camera_blocked_start is None:
            camera_blocked_start = current_time

        if current_time - camera_blocked_start >= 1.0:
            violation = "Camera Blocked"
            risk_score = 50
            risk_level = "High"
    else:
        camera_blocked_start = None

    # -----------------------------
    # Dummy placeholders for team integration
    # Later these will come from Members 1, 2, and 3
    # -----------------------------
    face_alert = False
    phone_alert = False
    gaze_alert = False

    final_score = risk_score

    if face_alert:
        final_score += 30

    if phone_alert:
        final_score += 50

    if gaze_alert:
        final_score += 20

    # -----------------------------
    # Fixed Overall Status Logic
    # -----------------------------
    if final_score >= 50:
        overall_status = "Violation"
    elif final_score >= 20 or violation != "Normal":
        overall_status = "Warning"
    else:
        overall_status = "Normal"

    if violation != "Normal":
        violation_count += 1

    # Draw bounding box around largest moving region
    if largest_box is not None:
        x, y, w, h, area = largest_box
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    latency_ms = (time.time() - frame_start_time) * 1000

    # -----------------------------
    # Text overlays on output video
    # -----------------------------
    cv2.putText(frame, f"Status: {violation}", (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.putText(frame, f"Overall: {overall_status}", (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    cv2.putText(frame, f"Risk Score: {final_score} ({risk_level})", (30, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Motion Ratio: {motion_ratio:.3f}", (30, 160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Latency: {latency_ms:.2f} ms", (30, 200),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Violation Count: {violation_count}", (30, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Save log only when a violation happens
    if violation != "Normal":
        violation_logs.append({
            "timestamp_seconds": round(current_time, 2),
            "frame_number": frame_number,
            "violation_type": violation,
            "overall_status": overall_status,
            "risk_score": final_score,
            "risk_level": risk_level,
            "motion_ratio": round(motion_ratio, 4),
            "frame_difference_score": round(frame_diff_score, 4),
            "brightness": round(brightness, 2),
            "latency_ms": round(latency_ms, 2)
        })

    out.write(frame)

cap.release()
out.release()

total_time = time.time() - start_time_total
processed_fps = frame_number / total_time

df = pd.DataFrame(violation_logs)
df.to_csv(log_path, index=False)

print("Processing complete")
print("Input video:", video_path)
print("Total frames processed:", frame_number)
print("Average FPS:", round(processed_fps, 2))
print("Total violations:", violation_count)
print("Violation log saved as:", log_path)
print("Output video saved as:", output_path)

Processing complete
Input video: normal_sitting.mp4
Total frames processed: 453
Average FPS: 16.86
Total violations: 0
Violation log saved as: normal_sitting_member4_log.csv
Output video saved as: normal_sitting_member4_output.mp4


In [10]:
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Uploaded video:", video_path)

Saving camera_blocked.mov to camera_blocked.mov
Uploaded video: camera_blocked.mov


In [11]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Video could not be opened")

fps_input = cap.get(cv2.CAP_PROP_FPS)
if fps_input == 0:
    fps_input = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

base_name = os.path.splitext(video_path)[0]

output_path = base_name + "_member4_output.mp4"
log_path = base_name + "_member4_log.csv"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps_input, (width, height))

background_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

previous_gray = None
frame_number = 0
violation_logs = []

excessive_start = None
camera_blocked_start = None
violation_count = 0

start_time_total = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_start_time = time.time()
    frame_number += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (21, 21), 0)

    fg_mask = background_subtractor.apply(blur)

    _, fg_mask = cv2.threshold(fg_mask, 200, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5, 5), np.uint8)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_DILATE, kernel)

    frame_diff_score = 0

    if previous_gray is not None:
        diff = cv2.absdiff(previous_gray, blur)
        _, diff_thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
        frame_diff_score = np.sum(diff_thresh > 0) / (width * height)

    previous_gray = blur.copy()

    contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    total_motion_area = 0
    largest_box = None

    for contour in contours:
        area = cv2.contourArea(contour)

        if area > 800:
            x, y, w, h = cv2.boundingRect(contour)
            total_motion_area += area

            if largest_box is None or area > largest_box[4]:
                largest_box = (x, y, w, h, area)

    motion_ratio = total_motion_area / (width * height)
    brightness = np.mean(gray)

    violation = "Normal"
    risk_score = 0
    risk_level = "Low"

    current_time = frame_number / fps_input

    # -----------------------------
    # Excessive Movement Detection
    # -----------------------------
    if motion_ratio > 0.08 or frame_diff_score > 0.07:
        if excessive_start is None:
            excessive_start = current_time

        if current_time - excessive_start >= 0.5:
            violation = "Excessive Movement Detected"

            if motion_ratio > 0.20:
                risk_score = 40
                risk_level = "High"
            elif motion_ratio > 0.12:
                risk_score = 25
                risk_level = "Medium"
            else:
                risk_score = 15
                risk_level = "Low"
    else:
        excessive_start = None

    # -----------------------------
    # Camera Blocked Detection
    # -----------------------------
    if brightness < 35:
        if camera_blocked_start is None:
            camera_blocked_start = current_time

        if current_time - camera_blocked_start >= 1.0:
            violation = "Camera Blocked"
            risk_score = 50
            risk_level = "High"
    else:
        camera_blocked_start = None

    # -----------------------------
    # Dummy placeholders for team integration
    # Later these will come from Members 1, 2, and 3
    # -----------------------------
    face_alert = False
    phone_alert = False
    gaze_alert = False

    final_score = risk_score

    if face_alert:
        final_score += 30

    if phone_alert:
        final_score += 50

    if gaze_alert:
        final_score += 20

    # -----------------------------
    # Fixed Overall Status Logic
    # -----------------------------
    if final_score >= 50:
        overall_status = "Violation"
    elif final_score >= 20 or violation != "Normal":
        overall_status = "Warning"
    else:
        overall_status = "Normal"

    if violation != "Normal":
        violation_count += 1

    # Draw bounding box around largest moving region
    if largest_box is not None:
        x, y, w, h, area = largest_box
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    latency_ms = (time.time() - frame_start_time) * 1000

    # -----------------------------
    # Text overlays on output video
    # -----------------------------
    cv2.putText(frame, f"Status: {violation}", (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.putText(frame, f"Overall: {overall_status}", (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    cv2.putText(frame, f"Risk Score: {final_score} ({risk_level})", (30, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Motion Ratio: {motion_ratio:.3f}", (30, 160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Latency: {latency_ms:.2f} ms", (30, 200),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.putText(frame, f"Violation Count: {violation_count}", (30, 240),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Save log only when a violation happens
    if violation != "Normal":
        violation_logs.append({
            "timestamp_seconds": round(current_time, 2),
            "frame_number": frame_number,
            "violation_type": violation,
            "overall_status": overall_status,
            "risk_score": final_score,
            "risk_level": risk_level,
            "motion_ratio": round(motion_ratio, 4),
            "frame_difference_score": round(frame_diff_score, 4),
            "brightness": round(brightness, 2),
            "latency_ms": round(latency_ms, 2)
        })

    out.write(frame)

cap.release()
out.release()

total_time = time.time() - start_time_total
processed_fps = frame_number / total_time

df = pd.DataFrame(violation_logs)
df.to_csv(log_path, index=False)

print("Processing complete")
print("Input video:", video_path)
print("Total frames processed:", frame_number)
print("Average FPS:", round(processed_fps, 2))
print("Total violations:", violation_count)
print("Violation log saved as:", log_path)
print("Output video saved as:", output_path)

Processing complete
Input video: camera_blocked.mov
Total frames processed: 220
Average FPS: 7.88
Total violations: 176
Violation log saved as: camera_blocked_member4_log.csv
Output video saved as: camera_blocked_member4_output.mp4


In [12]:
df = pd.read_csv(log_path)
df.head(), len(df)

(   timestamp_seconds  frame_number  violation_type overall_status  risk_score  \
 0               1.08            18  Camera Blocked      Violation          50   
 1               1.14            19  Camera Blocked      Violation          50   
 2               1.20            20  Camera Blocked      Violation          50   
 3               1.26            21  Camera Blocked      Violation          50   
 4               1.32            22  Camera Blocked      Violation          50   
 
   risk_level  motion_ratio  frame_difference_score  brightness  latency_ms  
 0       High           0.0                     0.0       30.31      130.34  
 1       High           0.0                     0.0       30.27      244.53  
 2       High           0.0                     0.0       30.06      209.03  
 3       High           0.0                     0.0       28.85      109.88  
 4       High           0.0                     0.0       27.75       68.04  ,
 176)